In [1]:
!pip install -U transformers

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("andrewmos/indian-legal-summaries-finetuned")
model = AutoModelForCausalLM.from_pretrained("andrewmos/indian-legal-summaries-finetuned")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=2040)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

/home/estudiante/tldr-uniandes/encoders-vs-decoders-classification/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Hi there! I’m a large language model, created by the Gemma team at Google DeepMind. I’m designed to take text and images as input and generate text as output. Think of me as an AI assistant that's trained to communicate and generate different kinds of creative text formats, like poems, code, scripts, musical pieces, email, letters, etc. 

I'm still under development, but I'm learning new things every day! 

---

To help me understand you better, you could also tell me:

*   **What's on your mind?** What would you like to know or talk about?
*   **What's the context?** Is there a specific situation or topic you're interested in?<end_of_turn>


In [3]:
!pip install rouge_score sacrebleu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
import torch
import os
from rouge_score import rouge_scorer
import sacrebleu
from unsloth import FastLanguageModel

# --------------------------------------------
# CONFIG
# --------------------------------------------
jsonl_file = "summaries_con_finetuning.jsonl"
save_every = 5  # Guardar cada 5 predicciones (aunque solo haya 3 se guarda al final)

INSTRUCTION = (
    "Provide a concise and accurate summary of the following legal judgment. "
    "Focus on the key facts, the legal reasoning, and the final verdict."
)

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# --------------------------------------------
# Cargar dataset test
# --------------------------------------------
dataset_eval = load_dataset(
    "andrewmos/indian-legal-summaries-alpaca-format",
    split="test"
)

print(f"Total test samples originales: {len(dataset_eval)}")

# SOLO TOMAR LAS 3 PRIMERAS FILAS
# dataset_eval = dataset_eval.select(range(3))
# print(f"Ejecutando solo con: {len(dataset_eval)} ejemplos (modo prueba)")

# --------------------------------------------
# Preparar modelo
# --------------------------------------------
FastLanguageModel.for_inference(model)
scorer = rouge_scorer.RougeScorer(['rouge2', 'rougeL'], use_stemmer=True)

# Para acumular métricas
all_metrics = []
temp_buffer = []  # <-- para guardar cada 5 antes de escribir

# Borrar archivo JSONL si existe
if os.path.exists(jsonl_file):
    os.remove(jsonl_file)

# --------------------------------------------
# LOOP
# --------------------------------------------
for i, row in enumerate(tqdm(dataset_eval)):

    row_id = row["id"]
    row_input = row["input"]
    row_reference = row["output"]

    # Construcción del prompt alpaca
    prompt = alpaca_prompt.format(INSTRUCTION, row_input, "")

    # Tokenización
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Generación
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Extraer predicción
    if "### Response:" in decoded:
        prediction = decoded.split("### Response:")[-1].strip()
    else:
        prediction = decoded.strip()

    # --------------------------------------------
    # MÉTRICAS
    # --------------------------------------------
    rouge = scorer.score(row_reference, prediction)
    rouge2 = rouge["rouge2"].fmeasure
    rougel = rouge["rougeL"].fmeasure

    bleu = sacrebleu.corpus_bleu([prediction], [[row_reference]]).score / 100

    avg = (rouge2 + rougel + bleu) / 3

    all_metrics.append({
        "id": row_id,
        "rouge2": rouge2,
        "rougeL": rougel,
        "bleu": bleu,
        "avg": avg
    })

    # --------------------------------------------
    # GUARDAR EN BUFFER
    # --------------------------------------------
    temp_buffer.append(
        '{"ID": "' + row_id + '", "Summary": "' + prediction.replace('"', "'") + '"}\n'
    )

    # --------------------------------------------
    # GUARDAR CADA save_every
    # (para 3 ejemplos no ejecutará este if)
    # --------------------------------------------
    if (i + 1) % save_every == 0:
        with open(jsonl_file, "a", encoding="utf-8") as f:
            for line in temp_buffer:
                f.write(line)
        temp_buffer = []

# --------------------------------------------
# GUARDAR LO QUE FALTE AL FINAL
# --------------------------------------------
if len(temp_buffer) > 0:
    with open(jsonl_file, "a", encoding="utf-8") as f:
        for line in temp_buffer:
            f.write(line)

# --------------------------------------------
# MÉTRICAS
# --------------------------------------------
df_metrics = pd.DataFrame(all_metrics)

print("\n📊 MÉTRICAS (solo 3 muestras)")
print(df_metrics)
print("\n📄 Archivo generado:", jsonl_file)

/tmp/ipykernel_2964771/87677140.py:8: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


Total test samples originales: 240



  0%|                                                   | 0/240 [00:00<?, ?it/s]

...


 12%|█████                                   | 30/240 [17:39<2:31:21, 43.25s/it]


 13%|█████▏                                  | 31/240 [18:11<2:18:05, 39.64s/it]


 13%|█████▎                                  | 32/240 [18:39<2:05:26, 36.18s/it]


 13%|█████▎                                  | 32/240 [18:39<2:01:17, 34.99s/it]

OutOfMemoryError: CUDA out of memory. Tried to allocate 17.53 GiB. GPU 0 has a total capacity of 23.78 GiB of which 14.20 GiB is free. Including non-PyTorch memory, this process has 9.21 GiB memory in use. Of the allocated memory 5.36 GiB is allocated by PyTorch, and 3.53 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)